In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [21]:
import warnings

# Suppress all pandas warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore")

In [22]:
import plotly.io as pio
pio.renderers.default = "vscode"

# Maximize the NOT NULL Values

In [23]:
def column_transformations(df, columns_list):
    # Keep rows with missing values in 'price', 'bed', or 'bath' while applying conditions

    # Remove rows where the 'price' <= 1000 and greater than the max allowed value (but keep NaNs)
    if 'price' in columns_list:
        df = df[((df['price'] > 1000) & (df['price'] <= (285 * 10**6))) | df['price'].isna()]

    # Remove rows where 'bed' > 25 (but keep NaNs)
    if 'bed' in columns_list:
        df = df[(df['bed'] <= 25) | df['bed'].isna()]

    # Remove rows where 'bath' > 20 (but keep NaNs)
    if 'bath' in columns_list:
        df = df[(df['bath'] <= 20) | df['bath'].isna()]
    
    # Reset index after filtering
    df = df.reset_index(drop=True)

    return df

In [24]:
def get_unique_city(x):
    unique_vals = x.dropna().unique()
    return unique_vals[0] if len(unique_vals) == 1 else np.nan

def get_mode_zip(x):
    vc = x.value_counts(normalize=True)
    
    if not vc.empty and vc.iloc[0] > 0.95:
        return vc.index[0]
    return np.nan

def dynamic_bins(group, na_column, compare_column, conf_count_per_bin = 20):
    """Create bins based on quantiles for each (zip_code, city, state) group."""

    prices = group[compare_column].dropna()
    if len(prices) == 0:
        group['range'] = 'All values are NaN'
        group['median'] = np.nan

        return group
        
    quantiles = np.quantile(prices, [0, 0.25, 0.5, 0.75, 1])
    quantiles = np.sort(np.unique(quantiles))
    
    if len(quantiles) == 1:
        group['range'] = f'Constant {quantiles[0]:.2f}'
        if group[na_column].notna().sum() >= conf_count_per_bin:
            group['median'] = group[na_column].median()
        else:
            group['median'] = np.nan

        return group

    # Define the bins
    bin_labels = [f"{quantiles[i]:.2f}-{quantiles[i+1]:.2f}" for i in range(len(quantiles)-1)]
    
    # Create the bins using pd.cut
    group['range'] = pd.cut(group[compare_column], bins=quantiles, labels=bin_labels,
                            include_lowest=True, duplicates='drop')

    # Assign the median value to each bin if the bin has more than 10 values, otherwise NaN
    median_values = group.groupby('range', observed=True)[na_column].apply(lambda x: x.median() if x.notna().sum() >= conf_count_per_bin else np.nan)
    
    # Map the calculated medians to the 'range' column
    group['median'] = group['range'].map(median_values)

    return group


def fill_missing_values_with_comparison(df, na_column, compare_column, conf_count_per_bin = 25):

    grouped_df = df.groupby(['zip_code', 'city', 'state'], group_keys=False)[df.columns.tolist()].apply(
        dynamic_bins, na_column=na_column, compare_column=compare_column, conf_count_per_bin=conf_count_per_bin
    )
    
    grouped_df[na_column] = grouped_df[na_column].fillna(grouped_df['median'])
    grouped_df = grouped_df.drop(['median', 'range'], axis=1)
    return grouped_df


def pipeline(df, columns_list):
    # state
    if 'state' in columns_list:
        df = df.dropna(subset=['state'])

    # city
    if 'city' in columns_list:
        df.loc[:, "city"] = df["city"].fillna(
            df.groupby(["state", "zip_code"])["city"].transform(get_unique_city)
        )
        df = df.dropna(subset=["city"])

    # zip_code
    if 'zip_code' in columns_list:
        df['zip_code'] = df['zip_code'].fillna(
            df.groupby(['city', 'state'])['zip_code'].transform(get_mode_zip)
        )
        df = df.dropna(subset=['zip_code'])

    # drop prev_sold_date, street
    if 'street' in columns_list:
        df = df.drop(['street'], axis=1)

    if 'prev_sold_date' in columns_list:
        df = df.drop(['prev_sold_date'], axis=1)

    # brokered_by
    if 'brokered_by' in columns_list:
        df["brokered_by"] = df["brokered_by"].fillna(999999)

    # For the below can also do some analysis by incorporating Feature A and Feature B relevant to 
    # Feature C (in which the NULL values have to be filled) and use techniques like Linear Regression to 
    # get an estimate but with this if we want to get an estimates for particular zip_codes + state + city 
    # the number of actual values available might be low to get an estimate in some of the groups. But can 
    # solve it similarly like quantiles by using confidence value

    # bed
    if 'bed' in columns_list:
        df = fill_missing_values_with_comparison(df.copy(), 'bed', 'house_size', 25)

    # bath
    if 'bath' in columns_list:
        df = fill_missing_values_with_comparison(df.copy(), 'bath', 'house_size', 25)

    # price (will be better if use linear reg to estimate the coeffs)
    if 'price' in columns_list:
        df['total_area_sqft'] = df['house_size'] + df['acre_lot'] * 43560
        df = fill_missing_values_with_comparison(df.copy(), 'price', 'total_area_sqft', 25)

        # drop
        df = df.drop(['total_area_sqft'], axis=1)

    not_required_columns = set(df.columns) - set(columns_list)
    df = df.drop(not_required_columns, axis=1)

    # drop na from all other columns
    df = df.dropna()

    # reset index
    df = df.reset_index(drop=True)
    return df

In [25]:
realtor_data = pd.read_csv('/kaggle/input/usa-real-estate-dataset/realtor-data.zip.csv')

In [26]:
columns_list = ['price', 'bed', 'state']

pipeline(column_transformations(realtor_data, columns_list), columns_list)

,price,bed,state
0,105000.0,3.0,Puerto Rico
1,80000.0,4.0,Puerto Rico
2,67000.0,2.0,Puerto Rico
3,145000.0,4.0,Puerto Rico
4,65000.0,6.0,Puerto Rico
...,...,...,...
1754400,359900.0,4.0,Washington
1754401,350000.0,3.0,Washington
1754402,440000.0,6.0,Washington
1754403,179900.0,2.0,Washington


# Hypothesis Testing

## Is there a significant difference in the housing prices between different cities or regions?

Dallas, San Antonio, Fort Worth - Texas

In [27]:
data = pd.read_csv('/kaggle/input/usa-real-estate-dataset/realtor-data.zip.csv')

columns_list = ['price', 'city', 'state']
data_preprocessed = pipeline(column_transformations(data, columns_list), columns_list)

In [28]:
data_preprocessed.head()

,price,city,state
0,105000.0,Adjuntas,Puerto Rico
1,80000.0,Adjuntas,Puerto Rico
2,67000.0,Juana Diaz,Puerto Rico
3,145000.0,Ponce,Puerto Rico
4,65000.0,Mayaguez,Puerto Rico


In [29]:
# Create a new column 'city-state' by combining 'city' and 'state'. To avoid same city name clashes across states

data_preprocessed['city-state'] = data_preprocessed['city'] + '-' + data_preprocessed['state']

In [30]:
data_preprocessed['city-state'].value_counts()

city-state
Houston-Texas                          23602
Chicago-Illinois                       18223
New York City-New York                 12632
Philadelphia-Pennsylvania              10372
Miami-Florida                           9619
                                       ...  
Hungry Horse-Montana                       1
Nordic Way-Montana                         1
Seeley Lake River Watch Trl-Montana        1
East Hope-Idaho                            1
Kahlotus-Washington                        1
Name: count, Length: 30513, dtype: int64

In [31]:
considered_cities = [
    'Dallas-Texas',
    'San Antonio-Texas',
    'Fort Worth-Texas'
]

In [32]:
import plotly.express as px

for city in considered_cities:
    fig = px.histogram(
        data_preprocessed[data_preprocessed['city-state'] == city]['price'],
        nbins=50,
        title=f"Distribution of Housing Prices in {city}",
        labels={'price': 'Price'}
    )
    fig.show()

In [33]:
import numpy as np
import plotly.express as px
from scipy.stats import norm

for city in considered_cities:
    prices = data_preprocessed[data_preprocessed['city-state'] == city]['price'].dropna()
    n = len(prices)
    if n < 2:
        continue  # Skip if not enough data
    prices_sorted = np.sort(prices)
    # Calculate theoretical quantiles for normal distribution
    theoretical_quantiles = norm.ppf((np.arange(1, n + 1) - 0.5) / n, loc=np.mean(prices_sorted), scale=np.std(prices_sorted))
    qq_df = pd.DataFrame({'Theoretical Quantiles': theoretical_quantiles, 'Sample Quantiles': prices_sorted})
    fig = px.scatter(
        qq_df,
        x='Theoretical Quantiles',
        y='Sample Quantiles',
        title=f"Q-Q Plot for Housing Prices in {city}",
        labels={'Theoretical Quantiles': 'Theoretical Quantiles (Normal)', 'Sample Quantiles': 'Sample Quantiles (Prices)'}
    )
    # Add reference line
    fig.add_shape(
        type='line',
        x0=qq_df['Theoretical Quantiles'].min(),
        y0=qq_df['Sample Quantiles'].min(),
        x1=qq_df['Theoretical Quantiles'].max(),
        y1=qq_df['Sample Quantiles'].max(),
        line=dict(color='red', dash='dash')
    )
    fig.show()

In [34]:
from scipy.stats import kstest

for city in considered_cities:
    prices = data_preprocessed[data_preprocessed['city-state'] == city]['price'].dropna()
    if len(prices) > 5000:
        stat, p_value = kstest(prices, 'norm', args=(prices.mean(), prices.std()))
        print(f"{city}: KS statistic={stat:.4f}, p-value={p_value:.4f}")
        if p_value > 0.05:
            print(f"  -> Prices in {city} are likely normal (fail to reject H0)")
        else:
            print(f"  -> Prices in {city} are NOT normal (reject H0)")

Dallas-Texas: KS statistic=0.2631, p-value=0.0000
  -> Prices in Dallas-Texas are NOT normal (reject H0)
San Antonio-Texas: KS statistic=0.2058, p-value=0.0000
  -> Prices in San Antonio-Texas are NOT normal (reject H0)
Fort Worth-Texas: KS statistic=0.2075, p-value=0.0000
  -> Prices in Fort Worth-Texas are NOT normal (reject H0)


In [35]:
from scipy.stats import kruskal

groups = [data_preprocessed[data_preprocessed['city-state'] == city]['price'].dropna()
          for city in considered_cities]
stat, p_value = kruskal(*groups)
print(f"Kruskal-Wallis statistic: {stat:.4f}, p-value: {p_value:.4f}")
if p_value < 0.05:
    print("Reject H0: Prices differ across cities.")
else:
    print("Fail to reject H0: No significant difference.")

Kruskal-Wallis statistic: 642.9030, p-value: 0.0000
Reject H0: Prices differ across cities.


## Do houses with more than 3 bedrooms significantly differ in price compared to houses with fewer bedrooms?

In [ ]:
data = pd.read_csv('/kaggle/input/usa-real-estate-dataset/realtor-data.zip.csv')

columns_list = ['bed', 'price']
data_preprocessed = pipeline(column_transformations(data, columns_list), columns_list)

In [ ]:
data_preprocessed['bedroom_category'] = data_preprocessed['bed'].apply(lambda x: 'More than 3' if x > 3 else '3 or fewer')

In [ ]:
data_preprocessed['bedroom_category'].value_counts()

bedroom_category
3 or fewer     1136478
More than 3     617927
Name: count, dtype: int64

In [ ]:
data_preprocessed.groupby('bedroom_category')['price'].describe()

,count,mean,std,min,25%,50%,75%,max
bedroom_category,,,,,,,,
3 or fewer,1136478.0,431903.298346,6.546932e+05,1100.0,199000.0,315000.0,490000.0,281500000.0
More than 3,617927.0,848084.792589,1.863433e+06,1300.0,342900.0,500000.0,799900.0,250000000.0


In [ ]:
import plotly.express as px

considered_bedroom_categories = ['3 or fewer', 'More than 3']

for cat in considered_bedroom_categories:
    fig = px.histogram(
        data_preprocessed[data_preprocessed['bedroom_category'] == cat]['price'],
        nbins=10,
        title=f"Distribution of Housing Prices in {cat} Bedrooms",
        labels={'price': 'Price'}
    )
    fig.show()

: 

: 

In [ ]:
import numpy as np
import plotly.express as px
from scipy.stats import norm

for bed_cat in considered_bedroom_categories:
    prices = data_preprocessed[data_preprocessed['bedroom_category'] == bed_cat]['price'].dropna()
    n = len(prices)
    if n < 2:
        continue  # Skip if not enough data
    prices_sorted = np.sort(prices)
    # Calculate theoretical quantiles for normal distribution
    theoretical_quantiles = norm.ppf((np.arange(1, n + 1) - 0.5) / n, loc=np.mean(prices_sorted), scale=np.std(prices_sorted))
    qq_df = pd.DataFrame({'Theoretical Quantiles': theoretical_quantiles, 'Sample Quantiles': prices_sorted})
    fig = px.scatter(
        qq_df,
        x='Theoretical Quantiles',
        y='Sample Quantiles',
        title=f"Q-Q Plot for Housing Prices in {bed_cat} Bedrooms",
        labels={'Theoretical Quantiles': 'Theoretical Quantiles (Normal)', 'Sample Quantiles': 'Sample Quantiles (Prices)'}
    )
    # Add reference line
    fig.add_shape(
        type='line',
        x0=qq_df['Theoretical Quantiles'].min(),
        y0=qq_df['Sample Quantiles'].min(),
        x1=qq_df['Theoretical Quantiles'].max(),
        y1=qq_df['Sample Quantiles'].max(),
        line=dict(color='red', dash='dash')
    )
    fig.show()

In [ ]:
from scipy.stats import kstest

for bed_cat in considered_bedroom_categories:
    prices = data_preprocessed[data_preprocessed['bedroom_category'] == bed_cat]['price'].dropna()
    if len(prices) > 5000:
        stat, p_value = kstest(prices, 'norm', args=(prices.mean(), prices.std()))
        print(f"{bed_cat}: KS statistic={stat:.4f}, p-value={p_value:.4f}")
        if p_value > 0.05:
            print(f"  -> Prices with {bed_cat} bedrooms are likely normal (fail to reject H0)")
        else:
            print(f"  -> Prices with {bed_cat} bedrooms are NOT normal (reject H0)")

3 or fewer: KS statistic=0.2649, p-value=0.0000
  -> Prices with 3 or fewer bedrooms are NOT normal (reject H0)
More than 3: KS statistic=0.3306, p-value=0.0000
  -> Prices with More than 3 bedrooms are NOT normal (reject H0)


In [ ]:
# Mann-Whitney U Test

from scipy.stats import mannwhitneyu

# Extract price data for each bedroom category
prices_few = data_preprocessed[data_preprocessed['bedroom_category'] == '3 or fewer']['price'].dropna()
prices_more = data_preprocessed[data_preprocessed['bedroom_category'] == 'More than 3']['price'].dropna()

# Mann-Whitney U Test (non-parametric test for independent samples)
stat, p_value = mannwhitneyu(prices_few, prices_more, alternative='two-sided')

print(f"Mann-Whitney U statistic: {stat:.4f}, p-value: {p_value:.4e}")
if p_value < 0.05:
    print("Reject H0: Significant difference in prices between bedroom categories.")
else:
    print("Fail to reject H0: No significant difference in prices between bedroom categories.")

Mann-Whitney U statistic: 209874333291.0000, p-value: 0.0000e+00
Reject H0: Significant difference in prices between bedroom categories.


# Statistical Measures

## What is the mean, median, and mode of housing prices in various cities or states?

In [ ]:
data = pd.read_csv('/kaggle/input/usa-real-estate-dataset/realtor-data.zip.csv')

columns_list = ['price', 'city', 'state']
data_preprocessed = pipeline(column_transformations(data, columns_list), columns_list)

In [ ]:
data_preprocessed['city-state'] = data_preprocessed['city'] + '-' + data_preprocessed['state']

In [ ]:
city_stats = data_preprocessed.groupby('city-state')['price'].agg(['mean', 'median', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan])
city_stats.columns = ['mean', 'median', 'mode']
city_stats

,mean,median,mode
city-state,,,
100 89 Lower Shepard Creek Road-North Carolina,1.900000e+06,1900000.0,1900000.0
139th Ave Unit Peck-Kansas,2.000000e+04,20000.0,20000.0
15th Ave Milton-Florida,1.700000e+04,17000.0,17000.0
177th Ave Wabasha-Minnesota,9.600000e+04,96000.0,96000.0
178th Ave Wabasha-Minnesota,8.250000e+04,82500.0,82500.0
...,...,...,...
Zumbro Falls-Minnesota,2.831538e+05,257500.0,225000.0
Zumbrota-Minnesota,2.970629e+05,291400.0,329900.0
Zuni-Virginia,4.449500e+05,444950.0,259900.0


## How do the price distributions of various cities compare (skewness, kurtosis)? 

In [ ]:

from scipy.stats import skew, kurtosis

city_stats = data_preprocessed.groupby('city-state')['price'].agg(
    mean='mean',
    median='median',
    mode=lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan,
    skewness=lambda x: skew(x, nan_policy='omit'),
    kurtosis=lambda x: kurtosis(x, nan_policy='omit')
)
city_stats

<ipython-input-36-62a572c269e3>:7: RuntimeWarning:

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.

<ipython-input-36-62a572c269e3>:8: RuntimeWarning:

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning:

invalid value encountered in greater

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning:

invalid value encountered in less

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning:

invalid value encountered in greater



,mean,median,mode,skewness,kurtosis
city-state,,,,,
100 89 Lower Shepard Creek Road-North Carolina,1.900000e+06,1900000.0,1900000.0,NaN,NaN
139th Ave Unit Peck-Kansas,2.000000e+04,20000.0,20000.0,NaN,NaN
15th Ave Milton-Florida,1.700000e+04,17000.0,17000.0,NaN,NaN
177th Ave Wabasha-Minnesota,9.600000e+04,96000.0,96000.0,NaN,NaN
178th Ave Wabasha-Minnesota,8.250000e+04,82500.0,82500.0,NaN,NaN
...,...,...,...,...,...
Zumbro Falls-Minnesota,2.831538e+05,257500.0,225000.0,0.425657,-0.559097
Zumbrota-Minnesota,2.970629e+05,291400.0,329900.0,0.848494,1.970860
Zuni-Virginia,4.449500e+05,444950.0,259900.0,0.000000,-2.000000


In [ ]:
data.columns

Index(['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street',
       'city', 'state', 'zip_code', 'house_size', 'prev_sold_date'],
      dtype='object')

## What is the correlation between the size of the lot (acre_lot) and the house price?


In [ ]:

data = pd.read_csv('/kaggle/input/usa-real-estate-dataset/realtor-data.zip.csv')

columns_list = ['price', 'acre_lot', 'house_size', 'bed', 'bath']
data_preprocessed = pipeline(column_transformations(data, columns_list), columns_list)

In [ ]:
data_preprocessed.corr()

,price,bed,bath,acre_lot,house_size
price,1.000000,0.253103,0.435088,0.010966,0.098596
bed,0.253103,1.000000,0.628383,0.000036,0.123326
bath,0.435088,0.628383,1.000000,-0.001268,0.155811
acre_lot,0.010966,0.000036,-0.001268,1.000000,0.001641
house_size,0.098596,0.123326,0.155811,0.001641,1.000000
